In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier

In [ ]:
data_dir = "dataset"

categories = ['healthy', 'miner', 'phoma', 'rust']

data = []
labels = []

In [ ]:
def get_features(img):
    try:
        # Resize to 128x128 (IMPORTANT)
        img = cv2.resize(img, (128, 128))

        # Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Flatten grayscale
        gray_flat = gray.flatten()

        # Histogram (256 bins)
        hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
        hist = hist.flatten()

        # Combine
        features = np.concatenate((gray_flat, hist))

        return features

    except:
        return None
for label, folder in enumerate(categories):
    path = os.path.join(data_dir, folder)

    for file in os.listdir(path):
        img_path = os.path.join(path, file)

        img = cv2.imread(img_path)
        if img is None:
            continue

        features = get_features(img)

        if features is None:
            continue

        data.append(features)
        labels.append(label)

x = np.array(data)
y = np.array(labels)

print("Dataset shape:", x.shape)

Dataset shape: (1264, 114688)


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y
)

In [ ]:
scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [ ]:
svm = SVC(
    kernel='rbf',
    C=10,
    probability=True,
    class_weight='balanced'
)

xgb = XGBClassifier(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.1,
    eval_metric='mlogloss'
)

model = VotingClassifier(
    estimators=[('svm', svm), ('xgb', xgb)],
    voting='soft'
)

In [ ]:
model.fit(x_train, y_train)

In [ ]:
pred = model.predict(x_test)

acc = accuracy_score(y_test, pred)

print("Accuracy:", acc)

In [ ]:
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))

print("Saved successfully")